# Hakeem — MedQA Conversational Finetune
**Base model:** `2kfi/MedGemma-4B-it-finetuned_V2.0`  
**Dataset:** `GBaker/MedQA-USMLE-4-options`  
**Style:** Conversation (chat template, train on responses only)  
**Target:** Push to `2kfi/` on Hugging Face Hub

## Step 1 — Install Dependencies

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2

## Step 2 — Load Your Finetuned Base Model

In [ ]:
from unsloth import FastVisionModel
import torch

model, processor = FastVisionModel.from_pretrained(
    "2kfi/MedGemma-4B-it-finetuned_V2.0",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)

print("Model loaded successfully.")

## Step 3 — Attach LoRA Adapters

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # MedQA is text-only, skip vision layers
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
    target_modules = "all-linear",
)

## Step 4 — Load Chat Template

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    processor,
    chat_template = "gemma-3",
)

## Step 5 — Load & Format MedQA as Conversations

Each sample becomes a proper chat turn:  
- **User:** question + labeled options (A/B/C/D)  
- **Assistant:** the correct answer with its label

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")

def format_as_conversation(example):
    """
    Converts a MedQA sample into a chat-style conversation.
    User asks the clinical question with options.
    Assistant responds with the correct answer.
    """
    # Format options as clean labeled list
    options_str = "\n".join([f"{k}: {v}" for k, v in example['options'].items()])

    user_message = (
        f"{example['question']}\n\n"
        f"Options:\n{options_str}"
    )

    # Find the label (A/B/C/D) that matches the correct answer text
    correct_label = next(
        (k for k, v in example['options'].items() if v == example['answer']),
        None
    )

    if correct_label:
        assistant_message = f"{correct_label}: {example['answer']}"
    else:
        assistant_message = example['answer']  # fallback

    # Apply the Gemma-3 chat template
    conversation = [
        {"role": "user",      "content": user_message},
        {"role": "assistant", "content": assistant_message},
    ]

    text = tokenizer.apply_chat_template(
        conversation,
        tokenize = False,
        add_generation_prompt = False,
    )

    # Strip BOS token — the trainer adds it
    text = text.lstrip('<bos>').lstrip('<|begin_of_text|>')

    return {"text": text}


dataset = raw_dataset.map(
    format_as_conversation,
    remove_columns = raw_dataset.column_names,
    load_from_cache_file = False,
)

print(f"Total samples: {len(dataset)}")
print("\n--- Sample conversation ---")
print(dataset[0]['text'])

## Step 6 — Configure Trainer

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = dataset,
    eval_dataset   = None,
    args = SFTConfig(
        dataset_text_field         = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch = 8
        warmup_steps               = 10,
        num_train_epochs           = 1,
        max_steps                  = -1,   # run full epoch
        learning_rate              = 2e-4,
        logging_steps              = 10,
        optim                      = "adamw_8bit",
        weight_decay               = 0.01,
        lr_scheduler_type          = "cosine",
        seed                       = 3407,
        output_dir                 = "hakeem_medqa_checkpoints",
        report_to                  = "none",
    ),
)

## Step 7 — Train on Responses Only
This masks the user turn so the model only learns to produce the assistant answer — not to memorize the question format.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)

## Step 8 — Train

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)

## Step 9 — Quick Sanity Check

In [ ]:
from transformers import TextStreamer

FastVisionModel.for_inference(model)

test_question = (
    "A 45-year-old woman presents with fatigue, weight gain, and cold intolerance. "
    "TSH is elevated and free T4 is low. What is the most appropriate treatment?\n\n"
    "Options:\nA: Methimazole\nB: Levothyroxine\nC: Propylthiouracil\nD: Radioactive iodine"
)

messages = [{"role": "user", "content": test_question}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize              = True,
    return_tensors        = "pt",
    return_dict           = True,
).to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens  = 64,
    temperature     = 1.0,
    top_p           = 0.95,
    top_k           = 64,
    streamer        = TextStreamer(tokenizer, skip_prompt=True),
)

## Step 10 — Push to Hugging Face Hub

**Important:** Store your HF token as a Colab Secret named `HF_TOKEN`, not hardcoded.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')  # Set this in Colab Secrets (key icon in sidebar)

model.push_to_hub_merged(
    "2kfi/Hakeem-MedGemma-4B-MedQA-v1",  # Change version tag as needed
    tokenizer,
    token = HF_TOKEN,
)

print("Upload complete.")